In [ ]:

import os
import json
import requests
import pandas as pd
from bs4 import BeautifulSoup

from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate


In [ ]:
# LLM Setup 

from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

llm = ChatOllama(
    model="phi3",
    temperature=0,
    timeout=30,
    streaming=False
)

prompt = PromptTemplate(
    input_variables=["text"],
    template="""
You are a financial bond data extraction expert.

Extract bond information from the text below.

Rules:
- Extract ONLY information explicitly present.
- Do NOT guess.
- If missing, return null.
- Return strictly valid JSON.

Return JSON in this exact format:

{{
  "bonds": [
    {{
      "ISIN": null,
      "Issuer_Name": null,
      "Country_of_Incorporation": null,
      "Country_of_Risk": null,
      "Sector": null,
      "State": null,
      "LEI": null,
      "CIK_Code": null,
      "Issuer_Incorporation_Year": null,
      "Issuer_Type": null,
      "Issue_Date": null,
      "Maturity_Date": null,
      "Issue_Currency": null,
      "Amount_Issued": null,
      "Coupon_Type": null,
      "Issuance_Coupon": null,
      "Interest_Payment_Frequency": null,
      "Instrument_Type": null,
      "Seniority": null,
      "Asset_Class": null
    }}
  ]
}}

Text:
{text}
"""
)

chain = prompt | llm


In [ ]:
def extract_text_from_url(url):

    headers = {
        "User-Agent": "Akshay akshay@email.com",  # use your real email
        "Accept-Encoding": "gzip, deflate",
        "Host": "www.sec.gov"
    }

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print("Failed to fetch page:", response.status_code)
        return ""

    soup = BeautifulSoup(response.text, "html.parser")

    for script in soup(["script", "style"]):
        script.extract()

    text = soup.get_text(separator=" ")
    return text.strip()


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def extract_bonds_from_url(url):

    print(f"\nProcessing URL: {url}")

    website_text = extract_text_from_url(url)

    if not website_text:
        print("No content extracted.")
        return []

    print("Total characters extracted:", len(website_text))

    # 🔥 Split large SEC page into chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=200
    )

    chunks = splitter.split_text(website_text)

    print("Total chunks:", len(chunks))

    all_bonds = []

    for chunk in chunks:

        try:
            # 🔎 Only send chunk if it contains financial keywords
            if not any(keyword in chunk.lower() for keyword in 
                       ["isin", "coupon", "maturity", "interest", "principal"]):
                continue

            response = chain.invoke({"text": chunk})

            result = json.loads(response.content)

            bonds = result.get("bonds", [])

            all_bonds.extend(bonds)

        except Exception as e:
            print("Chunk error:", e)

    return all_bonds


In [ ]:
urls = [
    "https://www.sec.gov/Archives/edgar/data/19617/000121390026015693/ea0276820-01_424b2.htm"
]

all_results = []

for url in urls:
    bonds = extract_bonds_from_url(url)
    all_results.extend(bonds)

if not all_results:
    print("⚠ No bond data extracted.")
else:
    df = pd.DataFrame(all_results)

    # Remove duplicates
    if "ISIN" in df.columns:
        df = df.drop_duplicates(subset=["ISIN"], keep="first")

    df = df.dropna(how="all")

    output_file = "bond_extracted_from_url.xlsx"
    df.to_excel(output_file, index=False)

    print("\n✅ Extraction Complete")
    print(f"Total Bonds Extracted: {len(df)}")
    print(f"Saved to: {output_file}")


In [ ]:
import os
os.getcwd()
